# Final v2 Retrain

Notebook ini membaca artefak hasil `python -m src.final_v2.run_final_v2`. Jalankan pipeline Python terlebih dahulu, lalu jalankan semua cell notebook ini untuk inspeksi hasil.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RESULTS = ROOT / 'results' / 'final_v2'
required = [
    RESULTS / 'final_v2_summary.json',
    RESULTS / 'final_v2_table.csv',
    RESULTS / 'front_history_final_v2.json',
    RESULTS / 'side_history_final_v2.json',
    RESULTS / 'fusion_final_v2_predictions.csv',
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('Artefak belum lengkap. Jalankan: python -m src.final_v2.run_final_v2\n' + '\n'.join(missing))

summary = json.loads((RESULTS / 'final_v2_summary.json').read_text(encoding='utf-8'))
table = pd.read_csv(RESULTS / 'final_v2_table.csv')
front_history = json.loads((RESULTS / 'front_history_final_v2.json').read_text(encoding='utf-8'))
side_history = json.loads((RESULTS / 'side_history_final_v2.json').read_text(encoding='utf-8'))
fusion_predictions = pd.read_csv(RESULTS / 'fusion_final_v2_predictions.csv')
table

## Ringkasan Protokol

In [ ]:
print('Scope:', summary['scope'])
print('\nFinal v1 reference:', summary['reference_final_v1'])
print('Exp21 stride30-best reference:', summary['reference_exp21_stride30_best'])
print('Reference comparison:', summary['reference_comparison'])

## Kurva Training

In [ ]:
def plot_history(history, title):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, ax1 = plt.subplots(figsize=(9, 4))
    ax1.plot(epochs, history['train_loss'], label='train loss')
    ax1.plot(epochs, history['val_loss'], label='val loss')
    ax1.set_xlabel('epoch')
    ax1.set_ylabel('loss')
    ax2 = ax1.twinx()
    ax2.plot(epochs, history['val_macro_f1'], color='tab:green', label='val Macro F1')
    ax2.set_ylabel('Macro F1')
    lines, labels = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines + lines2, labels + labels2, loc='lower right')
    ax1.set_title(title)
    ax1.grid(alpha=0.25)
    plt.show()

plot_history(front_history, 'Final v2 Front')
plot_history(side_history, 'Final v2 Side')

## Perbandingan Metrik

In [ ]:
display_cols = ['method', 'accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'safe_to_phone', 'phone_to_safe', 'generalization_status']
table[display_cols]

In [ ]:
ax = table.set_index('method')['f1_macro'].plot(kind='bar', figsize=(8, 4), color=['#3b6ea8', '#879957', '#c05a3b', '#7b5fa8'])
ax.axhline(summary['reference_final_v1']['average_fusion_f1_macro'], color='black', linestyle='--', linewidth=1, label='Final v1 fusion F1')
ax.axhline(summary['reference_exp21_stride30_best']['average_fusion_f1_macro'], color='tab:red', linestyle=':', linewidth=1.5, label='Exp21 stride30-best fusion F1')
ax.set_ylabel('Macro F1')
ax.set_title('Final v2 Macro F1')
ax.legend()
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()

## Confusion Matrix

In [ ]:
def plot_confusion(cm, title):
    fig, ax = plt.subplots(figsize=(4, 3.5))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks([0, 1], ['safe', 'phone'])
    ax.set_yticks([0, 1], ['safe', 'phone'])
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(title)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i][j], ha='center', va='center', color='black')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

plot_confusion(summary['front_single_view']['confusion_matrix'], 'Front final_v2')
plot_confusion(summary['side_single_view']['confusion_matrix'], 'Side final_v2')
plot_confusion(summary['fusion']['average_fusion']['confusion_matrix'], 'Average Fusion final_v2')
plot_confusion(summary['fusion']['adaptive_fusion']['confusion_matrix'], 'Adaptive Fusion final_v2')

## Statistical Tests

In [ ]:
stat_path = RESULTS / 'statistical_tests_vs_final_v1' / 'final_v1_vs_final_v2_statistics_summary.csv'
if stat_path.exists():
    stats_table = pd.read_csv(stat_path)
    display(stats_table)
else:
    print('Statistical tests belum ada. Jalankan: python -m src.final_v2.statistical_tests_final_v2_vs_final_v1')

## Prediction Preview

In [ ]:
fusion_predictions.head(10)